# Exploración de datos de Vitalidad — Madrid H3 res.9
**TFM – Álvaro Fernández-Uribarri Poveda · QuEA 2025–26**

Inventario y diagnóstico de todas las fuentes de datos disponibles para construir el índice de vitalidad urbana.

| Fuente | Variable | Archivo |
|--------|----------|----------|
| Censo de Locales (Ayto. Madrid) | Activación comercial | `200085-5-censo-locales.csv` |
| Padrón Histórico (Ayto. Madrid) | Población, extranjeros, edades | `padron_historico/padron_enero_XXXX.csv` |
| Censo 2021 INE | Tenencia, tamaño hogar, demografía | `censo2021_seccen_madrid.gpkg` |
| ADRH INE | Renta neta media | `31097.csv` |
| VUT (Ayto. Madrid) | Viviendas de uso turístico | `VIVIENDAS_USO_TURISTICO.xlsx` |
| Parques y jardines | Espacio verde | `200761-1-parques-jardines-geo.geo` |
| Mercados municipales | Atractores | `200967-4-mercados-csv.csv` |
| Terrazas | Activación exterior | `200085-6-censo-locales.csv` |

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)

BASE      = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA')
VITALIDAD = BASE / 'Vitalidad'
CRS = 'EPSG:25830'

# Inventario de archivos
print('=== INVENTARIO VITALIDAD ===')
total = 0
for root, dirs, files in os.walk(VITALIDAD):
    dirs[:] = [d for d in dirs if d not in ['imagenes', '__pycache__']]
    for f in sorted(files):
        if f.startswith('.'): continue
        path = Path(root) / f
        size = path.stat().st_size
        total += size
        rel = path.relative_to(VITALIDAD)
        print(f'  {size/1e6:7.1f} MB  {rel}')
print(f'\nTotal: {total/1e9:.2f} GB')

---
## 1. Censo de Locales — Activación comercial

In [ ]:
cols_loc = ['id_local', 'coordenada_x_local', 'coordenada_y_local',
            'id_tipo_acceso_local', 'desc_tipo_acceso_local',
            'id_situacion_local', 'desc_situacion_local',
            'id_seccion', 'desc_seccion',
            'id_division', 'desc_division',
            'id_epigrafe', 'desc_epigrafe',
            'id_distrito_local', 'desc_distrito_local']

print('Cargando Censo de Locales...')
loc = pd.read_csv(VITALIDAD / '200085-5-censo-locales.csv', sep=';',
                  usecols=cols_loc, low_memory=False)
print(f'Total registros: {len(loc):,}')
print()

# Situación de los locales
print('--- Situación de los locales ---')
sit = loc.groupby(['id_situacion_local','desc_situacion_local']).size().rename('n')
sit_df = sit.reset_index()
sit_df['%'] = (sit_df['n'] / len(loc) * 100).round(1)
print(sit_df.to_string(index=False))

In [ ]:
# Locales activos
activos = loc[loc['id_situacion_local'] == 1].copy()
print(f'Locales activos (Abierto): {len(activos):,}  ({len(activos)/len(loc)*100:.1f}%)')
print()

# Tipo de acceso
print('--- Tipo de acceso (activos) ---')
print(activos.groupby('desc_tipo_acceso_local').size().sort_values(ascending=False)
      .rename('n').to_frame().assign(pct=lambda df: (df['n']/len(activos)*100).round(1)).to_string())
print()

# Top 20 actividades
print('--- Top 20 actividades (activos) ---')
top20 = (activos['desc_epigrafe'].value_counts().head(20)
         .rename('n').to_frame()
         .assign(pct=lambda df: (df['n']/len(activos)*100).round(2)))
print(top20.to_string())

In [ ]:
# Secciones con más y menos locales activos
loc_x_secc = activos.groupby('id_seccion').size().rename('n_activos').sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución
ax = axes[0]
loc_x_secc.hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(loc_x_secc.median(), color='red', linestyle='--', label=f'Mediana={loc_x_secc.median():.0f}')
ax.set_xlabel('Nº locales activos por sección censal')
ax.set_ylabel('Nº secciones')
ax.set_title('Distribución locales activos/sección')
ax.legend()

# Top 15 divisiones
ax2 = axes[1]
div = (activos['desc_division'].value_counts().head(15))
div.plot(kind='barh', ax=ax2, color='steelblue')
ax2.set_title('Top 15 divisiones de actividad')
ax2.set_xlabel('Nº locales')
ax2.invert_yaxis()

# Tasa de activación por sección
ax3 = axes[2]
tasa_secc = loc.groupby('id_seccion').apply(
    lambda g: (g['id_situacion_local']==1).sum() / len(g)
).rename('tasa_activacion')
tasa_secc.hist(bins=40, ax=ax3, color='darkorange', edgecolor='white')
ax3.axvline(tasa_secc.median(), color='red', linestyle='--',
            label=f'Mediana={tasa_secc.median():.2f}')
ax3.set_xlabel('Tasa de activación (activos/total)')
ax3.set_title('Tasa de activación por sección')
ax3.legend()

plt.tight_layout()
Path('imagenes').mkdir(exist_ok=True)
plt.savefig('imagenes/eda_locales.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Estadísticas locales activos/sección:')
print(loc_x_secc.describe().round(1).to_string())

---
## 2. Terrazas

In [ ]:
terrazas = pd.read_csv(VITALIDAD / '200085-6-censo-locales.csv', sep=';', low_memory=False)
print(f'Terrazas: {len(terrazas):,} registros')
print(f'Columnas: {len(terrazas.columns)}')
print()
# Situación
if 'id_situacion_local' in terrazas.columns:
    print('Situación:')
    print(terrazas['id_situacion_local'].value_counts().to_string())
# Columnas clave
key_cols = [c for c in terrazas.columns if any(k in c.lower() for k in
            ['terraza','mesas','sillas','situacion','distrito','seccion','coordenada'])]
print(f'\nColumnas relevantes: {key_cols[:15]}')
print()
print(terrazas[key_cols[:8]].head(3).to_string())

---
## 3. Padrón Histórico 2015–2025 — Evolución poblacional

In [ ]:
padron_dir = VITALIDAD / 'padron_historico'
years = list(range(2015, 2026))

def load_padron(year):
    f = padron_dir / f'padron_enero_{year}.csv'
    # 2023-2025: UTF-8 BOM; 2015-2022: latin-1
    for enc in ['utf-8-sig', 'latin-1']:
        try:
            df = pd.read_csv(f, sep=';', encoding=enc, low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    # Normalizar nombres de columnas
    df.columns = [c.strip().upper().lstrip('﻿') for c in df.columns]
    # Normalizar columnas de población
    rename = {}
    for c in df.columns:
        if 'ESPANOL' in c and 'HOMBRE' in c: rename[c] = 'ESP_H'
        elif 'ESPANOL' in c and 'MUJER' in c: rename[c] = 'ESP_M'
        elif 'EXTRAN' in c and 'HOMBRE' in c: rename[c] = 'EXT_H'
        elif 'EXTRAN' in c and 'MUJER' in c: rename[c] = 'EXT_M'
    df = df.rename(columns=rename)
    for col in ['ESP_H','ESP_M','EXT_H','EXT_M']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['pob_total'] = df['ESP_H'] + df['ESP_M'] + df['EXT_H'] + df['EXT_M']
    df['extranjeros'] = df['EXT_H'] + df['EXT_M']
    df['year'] = year
    # CUSEC: algunos años (2024-25) tienen COD_DIST_SECCION como float por NaN → convertir via int
    if 'COD_DIST_SECCION' in df.columns:
        raw = (pd.to_numeric(df['COD_DIST_SECCION'], errors='coerce')
               .fillna(0).astype(int).astype(str).str.zfill(5))
        df['CUSEC'] = '28079' + raw
    return df

print('Cargando padron 2015-2025...')
padrones = {y: load_padron(y) for y in years}

# Agregado total Madrid por año
resumen = []
for y, df in padrones.items():
    resumen.append({
        'year': y,
        'pob_total': df['pob_total'].sum(),
        'extranjeros': df['extranjeros'].sum(),
        'n_secciones': df['CUSEC'].nunique() if 'CUSEC' in df.columns else np.nan
    })
res_df = pd.DataFrame(resumen)
res_df['pct_ext'] = res_df['extranjeros'] / res_df['pob_total'] * 100
print(res_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Población total
ax = axes[0]
ax.plot(res_df['year'], res_df['pob_total']/1e6, 'o-', color='steelblue', linewidth=2)
ax.set_title('Población total Madrid (enero)')
ax.set_ylabel('Millones')
ax.set_xlabel('Año')
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f M'))
ax.grid(True, alpha=0.4)

# % extranjeros
ax2 = axes[1]
ax2.plot(res_df['year'], res_df['pct_ext'], 'o-', color='darkorange', linewidth=2)
ax2.set_title('% Extranjeros (enero)')
ax2.set_ylabel('%')
ax2.set_xlabel('Año')
ax2.grid(True, alpha=0.4)

# Nº secciones
ax3 = axes[2]
ax3.plot(res_df['year'], res_df['n_secciones'], 'o-', color='green', linewidth=2)
ax3.set_title('Nº secciones censales')
ax3.set_ylabel('Secciones')
ax3.set_xlabel('Año')
ax3.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('imagenes/eda_padron_evolucion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Turnover proxy: variación relativa de población por sección entre 2015 y 2025
# Panel: población total por sección y año
panel = []
for y, df in padrones.items():
    agg = df.groupby('CUSEC')[['pob_total','extranjeros']].sum().reset_index()
    agg['year'] = y
    panel.append(agg)

panel_df = pd.concat(panel, ignore_index=True)
# Filtrar solo Madrid capital y excluir sección dummy '00000' (filas sin CUSEC real)
panel_df = panel_df[
    (panel_df['CUSEC'].str.startswith('28079')) &
    (panel_df['CUSEC'] != '2807900000') &
    (panel_df['pob_total'] > 0)
]

# Secciones con datos en todos los años
secciones_completas = (
    panel_df.groupby('CUSEC')['year'].count() == len(years)
)
secciones_ok = secciones_completas[secciones_completas].index
print(f'Secciones con datos en los {len(years)} anos: {len(secciones_ok)}')

# Ordenar por año para que iloc[0]=2015 e iloc[-1]=2025
panel_ok = (panel_df[panel_df['CUSEC'].isin(secciones_ok)]
            .sort_values(['CUSEC', 'year'])
            .reset_index(drop=True))

# Métrica de turnover: CV de la población (std/mean)
turnover_df = (
    panel_ok.groupby('CUSEC')['pob_total']
    .agg(pob_mean='mean', pob_std='std',
         pob_2015=lambda x: x.iloc[0],
         pob_2025=lambda x: x.iloc[-1])
    .assign(
        cv_pob=lambda df: df['pob_std'] / df['pob_mean'],
        cambio_pct=lambda df: (df['pob_2025'] - df['pob_2015']) / df['pob_2015'] * 100
    )
    .reset_index()
)

print(f'\nMetricas de variabilidad poblacional por seccion:')
print(turnover_df[['cv_pob','cambio_pct']].describe().round(3).to_string())

In [ ]:
# Evolución del % de extranjeros por sección (proxy de llegadas recientes)
ext_panel = (
    panel_ok.groupby('CUSEC').apply(
        lambda g: g.sort_values('year')['extranjeros'].values / 
                  g.sort_values('year')['pob_total'].values
    )
)

ext_2015 = panel_ok[panel_ok['year']==2015].set_index('CUSEC')['extranjeros'] / \
           panel_ok[panel_ok['year']==2015].set_index('CUSEC')['pob_total']
ext_2025 = panel_ok[panel_ok['year']==2025].set_index('CUSEC')['extranjeros'] / \
           panel_ok[panel_ok['year']==2025].set_index('CUSEC')['pob_total']

delta_ext = (ext_2025 - ext_2015).dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
delta_ext.hist(bins=50, ax=ax, color='darkorange', edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(delta_ext.median(), color='red', linestyle='--',
           label=f'Mediana={delta_ext.median():.3f}')
ax.set_xlabel('Δ % extranjeros (2025 - 2015)')
ax.set_title('Cambio en % extranjeros por sección')
ax.legend()

ax2 = axes[1]
turnover_df['cv_pob'].hist(bins=50, ax=ax2, color='steelblue', edgecolor='white')
ax2.axvline(turnover_df['cv_pob'].median(), color='red', linestyle='--',
            label=f'Mediana={turnover_df["cv_pob"].median():.3f}')
ax2.set_xlabel('CV población (std/mean 2015-2025)')
ax2.set_title('Variabilidad poblacional por sección (proxy turnover)')
ax2.legend()

plt.tight_layout()
plt.savefig('imagenes/eda_padron_turnover.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Censo 2021 — Demografía y tenencia por sección censal

In [ ]:
secc = gpd.read_file(str(VITALIDAD / 'censo2021_seccen_madrid.gpkg'))
print(f'Secciones censales Madrid: {len(secc)}')
print(f'Columnas: {list(secc.drop(columns="geometry").columns)}')
print()

vars_num = [c for c in secc.columns if c not in ['CUSEC','cod_distrito','cod_seccion',
            'nombre_municipio','ccaa','cpro','cmun','dist','secc','geometry']]
print('=== Estadísticas descriptivas ===')
print(secc[vars_num].describe().round(3).T[['mean','std','min','50%','max']].to_string())

In [ ]:
# Correlación entre variables del Censo 2021
corr_vars = ['pct_alquiler','pct_propiedad','tam_medio_hogar','pct_1persona',
             'pct_extranjeros','tasa_paro','pct_educ_superior','pct_65mas',
             'pct_viv_noppal','pob_total']
corr_vars = [v for v in corr_vars if v in secc.columns]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap de correlaciones
ax = axes[0]
corr = secc[corr_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.3, ax=ax, square=True)
ax.set_title('Correlación entre variables Censo 2021', fontsize=11)

# Distribuciones
ax2 = axes[1]
plot_vars = ['pct_alquiler','pct_extranjeros','tasa_paro','pct_educ_superior','pct_65mas']
for v in plot_vars:
    if v in secc.columns:
        secc[v].hist(bins=40, alpha=0.5, ax=ax2, label=v, density=True)
ax2.set_xlabel('Valor')
ax2.set_title('Distribución de variables clave (Censo 2021)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('imagenes/eda_censo2021.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mapas del Censo 2021
map_vars = [
    ('pct_alquiler',    '% viviendas en alquiler',   'YlOrRd'),
    ('pct_extranjeros', '% extranjeros (2021)',       'PuRd'),
    ('tasa_paro',       'Tasa de paro',               'Reds'),
    ('pct_educ_superior','% estudios superiores',    'Blues'),
    ('tam_medio_hogar', 'Tamaño medio del hogar',     'YlGn'),
    ('pct_65mas',       '% mayores de 64 años',       'Oranges'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (var, title, cmap) in zip(axes.flat, map_vars):
    if var not in secc.columns:
        ax.set_visible(False)
        continue
    vmax = secc[var].quantile(0.97)
    secc.plot(column=var, ax=ax, cmap=cmap, vmin=0, vmax=vmax,
              legend=True, legend_kwds={'shrink': 0.6, 'label': var},
              linewidth=0.05, edgecolor='none', missing_kwds={'color': '#EEEEEE'})
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()

plt.suptitle('Variables Censo 2021 por sección censal — Madrid capital', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('imagenes/eda_censo2021_mapas.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. ADRH — Renta neta media por sección (2015–2023)

In [ ]:
adrh = pd.read_csv(VITALIDAD / '31097.csv', sep=';', low_memory=False)

# Madrid capital, nivel sección, renta neta por persona
renta = adrh[
    (adrh['Municipios'].astype(str).str.startswith('28079')) &
    (adrh['Indicadores de renta media y mediana'] == 'Renta neta media por persona')
].dropna(subset=['Secciones']).copy()

renta['CUSEC'] = renta['Secciones'].astype(str).str[:10]
renta['renta'] = pd.to_numeric(renta['Total'], errors='coerce') * 1000
renta['year']  = renta['Periodo'].astype(int)

# Estadísticas por año
renta_yr = renta.groupby('year')['renta'].agg(['mean','median','std','count'])
renta_yr.columns = ['Media €', 'Mediana €', 'Std €', 'N secciones']
print('=== Renta neta media por persona — Madrid por año ===')
print(renta_yr.round(0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Evolución media
ax = axes[0]
ax.plot(renta_yr.index, renta_yr['Media €']/1000, 'o-', color='steelblue', linewidth=2, label='Media')
ax.plot(renta_yr.index, renta_yr['Mediana €']/1000, 's--', color='darkorange', linewidth=2, label='Mediana')
ax.fill_between(renta_yr.index,
                (renta_yr['Media €'] - renta_yr['Std €'])/1000,
                (renta_yr['Media €'] + renta_yr['Std €'])/1000,
                alpha=0.2, color='steelblue', label='± 1 std')
ax.set_title('Evolución renta neta/persona — Madrid')
ax.set_ylabel('Miles €')
ax.legend()
ax.grid(True, alpha=0.4)

# Distribución 2022
ax2 = axes[1]
renta22 = renta[renta['year']==2022]['renta'].dropna()
renta22.hist(bins=50, ax=ax2, color='steelblue', edgecolor='white')
ax2.axvline(renta22.median(), color='red', linestyle='--',
            label=f'Mediana = {renta22.median()/1000:.1f}k€')
ax2.set_xlabel('Renta neta/persona (€)')
ax2.set_title('Distribución por sección censal (2022)')
ax2.legend()

plt.tight_layout()
plt.savefig('imagenes/eda_renta.png', dpi=150, bbox_inches='tight')
plt.show()

# Datos faltantes
n_total = renta[renta['year']==2022]['CUSEC'].nunique()
print(f'\nSecciones con dato de renta (2022): {n_total} de 2443')
print(f'Cobertura: {n_total/2443*100:.1f}%')

---
## 6. Viviendas de Uso Turístico (VUT)

In [ ]:
vut = pd.read_excel(VITALIDAD / 'VIVIENDAS_USO_TURISTICO.xlsx', engine='openpyxl')
print(f'VUT: {len(vut):,} registros')
print(f'Columnas: {list(vut.columns)}')
print()
print(vut.head(3).to_string())
print()
print('Estadísticas básicas:')
print(vut.describe().T.to_string())

In [ ]:
# VUT por distrito / sección si está disponible
geo_cols = [c for c in vut.columns if any(k in c.upper() for k in
            ['DISTRITO','BARRIO','SECCION','COORD','LAT','LON','X','Y'])]
cat_cols  = [c for c in vut.columns if vut[c].dtype == 'object']
print(f'Columnas geográficas: {geo_cols}')
print(f'Columnas categoriales: {cat_cols}')
print()
for c in cat_cols[:4]:
    vc = vut[c].value_counts().head(10)
    print(f'{c}:\n{vc.to_string()}\n')

---
## 7. Parques, Jardines y Mercados

In [ ]:
# Parques
parques = gpd.read_file(str(VITALIDAD / '200761-1-parques-jardines-geo.geo'))
print(f'Parques y jardines: {len(parques)} registros')
print(f'CRS: {parques.crs}')
print(f'Columnas: {list(parques.columns)}')
print()
print(parques.drop(columns='geometry').describe(include='all').T.head(10).to_string())

In [ ]:
# Mercados
mercados = pd.read_csv(VITALIDAD / '200967-4-mercados-csv.csv', encoding='latin-1', sep=';')
print(f'Mercados municipales: {len(mercados)} registros')
print(f'Columnas: {list(mercados.columns)}')
print()
print(mercados.head(5).to_string())

In [ ]:
# Mapa combinado: parques + mercados
parques_utm = parques.to_crs(CRS) if parques.crs else parques

# Mercados a GeoDataFrame si tienen coordenadas
coord_cols = [c for c in mercados.columns if 'COORDENADA' in c.upper() or 'LATITUD' in c.upper() or 'X_COORD' in c.upper()]
print('Columnas coord. mercados:', coord_cols)

fig, ax = plt.subplots(figsize=(8, 8))
# Fondo secciones
secc.plot(ax=ax, color='#F0F0F0', edgecolor='#CCCCCC', linewidth=0.3)
# Parques
try:
    parques_utm.plot(ax=ax, color='#2ecc71', alpha=0.6, label='Parques')
except:
    pass
# Mercados
if len(coord_cols) >= 2:
    x_col = [c for c in coord_cols if 'X' in c.upper() or 'LON' in c.upper()][0]
    y_col = [c for c in coord_cols if 'Y' in c.upper() or 'LAT' in c.upper()][0]
    merc_valid = mercados.dropna(subset=[x_col, y_col])
    gdf_merc = gpd.GeoDataFrame(
        merc_valid,
        geometry=gpd.points_from_xy(merc_valid[x_col], merc_valid[y_col]),
        crs=CRS
    )
    gdf_merc.plot(ax=ax, color='red', markersize=20, zorder=5, label=f'Mercados (n={len(gdf_merc)})')
ax.set_title('Parques y mercados municipales — Madrid')
ax.set_axis_off()
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('imagenes/eda_parques_mercados.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Cobertura y datos faltantes — resumen

In [ ]:
# Resumen de cobertura de todas las fuentes a nivel sección censal
N_SECC = 2443  # secciones Madrid capital

cobertura = [
    ('Censo de Locales (activos)',    len(activos['id_seccion'].unique()),         N_SECC),
    ('Padrón 2025 (secciones)',       len(panel_ok[panel_ok['year']==2025]),       N_SECC),
    ('Censo 2021 (pct_alquiler)',     secc['pct_alquiler'].notna().sum(),          N_SECC),
    ('Censo 2021 (tam_hogar)',        secc['tam_medio_hogar'].notna().sum(),       N_SECC),
    ('ADRH renta 2022',               n_total,                                     N_SECC),
    ('Turnover (CV pob 2015-25)',     len(turnover_df),                            N_SECC),
]

print('=== COBERTURA POR FUENTE (sobre 2443 secciones Madrid) ===')
print(f'{"Fuente":<40}  {"N":>6}  {"Cobertura":>10}')
print('-' * 60)
for label, n, total in cobertura:
    pct = n / total * 100
    bar = '█' * int(pct/5) + '░' * (20 - int(pct/5))
    print(f'{label:<40}  {n:>6,}  {pct:>6.1f}%  {bar}')

print()
print('NOTA: Algunas secciones pueden tener secreto estadístico (INE) → NaN.')

In [ ]:
# Guardar tabla de turnover para el índice de vitalidad
turnover_save = turnover_df[['CUSEC','cv_pob','cambio_pct']].copy()
turnover_save.to_csv(VITALIDAD / 'turnover_padron_2015_2025.csv', index=False)
print(f'Guardado: turnover_padron_2015_2025.csv ({len(turnover_save)} secciones)')

# Guardar panel completo
panel_df.to_csv(VITALIDAD / 'panel_padron_2015_2025.csv', index=False)
print(f'Guardado: panel_padron_2015_2025.csv ({len(panel_df)} registros)')

---
## 9. Relación preliminar: morfología vs variables de vitalidad

In [ ]:
# Cargar morfología v10 y hacer join con secciones censales
gdf_morf = gpd.read_file(str(BASE / 'madrid_morfologia_h3_v10.gpkg')).to_crs(CRS)

# Join secciones → H3 (centroide de H3 dentro de sección)
cents = gpd.GeoDataFrame(geometry=gdf_morf.geometry.centroid, crs=CRS)
cents['hex_id'] = gdf_morf['hex_id'].values
cents['tipologia'] = gdf_morf['tipologia_label'].values

joined = gpd.sjoin(cents, secc[['CUSEC','pct_alquiler','tam_medio_hogar',
                                  'pct_extranjeros','tasa_paro','pct_educ_superior',
                                  'pct_65mas','pct_viv_noppal','geometry']],
                   how='left', predicate='within')

print('Variables demográficas por tipología morfológica:')
vars_show = ['pct_alquiler','tam_medio_hogar','pct_extranjeros','tasa_paro','pct_educ_superior','pct_65mas']
vars_show = [v for v in vars_show if v in joined.columns]
tabla = joined[joined['tipologia'].isin(['Espontáneo','Planificado'])].groupby('tipologia')[vars_show].mean().round(3)
print(tabla.to_string())

In [ ]:
# Boxplots: distribución de variables demográficas por tipología
df_plot = joined[joined['tipologia'].isin(['Espontáneo','Planificado'])].copy()
orden = ['Espontáneo', 'Planificado']
palette = {'Espontáneo': '#d7191c', 'Planificado': '#2b83ba'}

n_vars = len(vars_show)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, var in zip(axes.flat, vars_show):
    sns.boxplot(data=df_plot, x='tipologia', y=var, order=orden,
                palette=palette, ax=ax, width=0.5, fliersize=2)
    ax.set_title(var, fontsize=10)
    ax.set_xlabel('')

plt.suptitle('Variables demográficas por tipología morfológica (H3 centroides → sección)', fontsize=12)
plt.tight_layout()
plt.savefig('imagenes/eda_morfologia_vs_demo.png', dpi=150, bbox_inches='tight')
plt.show()